In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from claims_fraud.validation_tuning import (
    evaluate_cv,
    last_20_gain,
    load_data,
    make_logistic_pipeline,
    run_tuning,
    save_optimization_plot,
    summarize_validation,
    temporal_evaluation,
)

DATA = ROOT / "data" / "raw"
OUT = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)


## 1. Load the project data and enforce the leakage boundary

The model matrix excludes `investigation_opened`, `days_to_settle`, `amount_paid_xaf`, and `fraud_flag`. Duplicate claim rows are removed before modelling.

In [2]:
bundle = load_data(DATA)
print("Model rows:", len(bundle.model_data))
print("Features:", bundle.X.shape[1])
print("Fraud prevalence:", round(bundle.y.mean(), 4))


Model rows: 32176
Features: 31
Fraud prevalence: 0.0305


## 2. Validation before tuning

The first table is deliberately produced before the Optuna search.

- Random stratified 5-fold measures ordinary IID-style discrimination.
- Grouped 5-fold holds out whole garages, testing generalisation across the fraud-clustering entity.
- Temporal evaluation trains on development claims and evaluates January–June 2026.

All preprocessing is fitted inside each fold through the sklearn pipeline. Holder history for validation rows uses only the training fold as its source.

In [3]:
baseline = make_logistic_pipeline(class_weight="balanced")
random_df = evaluate_cv(bundle.X, bundle.y, baseline, bundle.model_data, "random")
grouped_df = evaluate_cv(
    bundle.X, bundle.y, baseline, bundle.model_data, "grouped",
    groups=bundle.model_data["garage_id"]
)
temporal_df = temporal_evaluation(bundle, baseline)

comparison = summarize_validation(random_df, grouped_df, temporal_df)
comparison


,scheme,PR-AUC,ROC-AUC,fraud prevalence
0,Random stratified 5-fold,0.1078 ± 0.0145,0.7748 ± 0.0156,0.0305
1,Grouped by garage 5-fold,0.0544 ± 0.0357,0.5740 ± 0.1328,0.0303
2,Temporal Jan-Jun 2026,0.1143,0.7539,0.0384


In [4]:
random_df.to_csv(OUT / "random_cv_folds.csv", index=False)
grouped_df.to_csv(OUT / "grouped_cv_folds.csv", index=False)
temporal_df.to_csv(OUT / "temporal_evaluation.csv", index=False)
comparison.to_csv(OUT / "validation_comparison.csv", index=False)


### Interpretation

Do not reconcile the three numbers into one. The grouped result is the critical stress test when fraud clusters by garage; the temporal result is the closest of these validation schemes to the production question of predicting later claims from earlier claims. The insurer-facing conclusion should be based on the validation scheme that best represents the deployment setting, with the other two retained as diagnostics.

## 3. Hyperparameter tuning

Only after the validation table exists, run the persistent Optuna study. The objective is **grouped 5-fold PR-AUC** because the brief specifically requires checking whether tuning gains survive grouped validation.

In [5]:
study = run_tuning(bundle, OUT / "optuna_xgb_grouped.db", n_trials=60)
print("Completed trials:", sum(t.state.name == "COMPLETE" for t in study.trials))
print("Best grouped PR-AUC:", study.best_value)
print("Best parameters:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")


Completed trials: 112
Best grouped PR-AUC: 0.055461573702525725
Best parameters:
n_estimators: 117
max_depth: 2
learning_rate: 0.013228163022535962
subsample: 0.9249793640915815
colsample_bytree: 0.7709439072317795
min_child_weight: 3
gamma: 0.00034851104050698245
reg_alpha: 0.02419176304474802
reg_lambda: 1.966000130617237
scale_pos_weight: 9.591945376180083


In [6]:
save_optimization_plot(study, OUT / "optuna_best_score_by_trial.png")
gain = last_20_gain(study)
gain


{'completed_trials': 112,
 'best_pr_auc': 0.055461573702525725,
 'best_before_final_20': 0.054087758105312723,
 'final_20_marginal_gain': 0.0013738155972130017}

In [7]:
pd.DataFrame([gain]).to_csv(OUT / "optuna_last_20_gain.csv", index=False)


In [8]:
from claims_fraud.validation_tuning import (
    collect_oof_predictions,
    compare_class_weighting,
    threshold_report,
)

# Class weighting comparison
imbalance_comparison = compare_class_weighting(
    bundle.X, bundle.y, bundle.model_data
)
imbalance_comparison


,class_weight,PR-AUC,ROC-AUC,precision_at_0_5,recall_at_0_5
0,none,0.052291,0.555545,0.050000,0.002235
1,balanced,0.054401,0.574032,0.037908,0.366581


In [9]:
# Threshold adjustment using grouped out-of-fold probabilities
oof = collect_oof_predictions(
    bundle.X, bundle.y, baseline, bundle.model_data,
    scheme="grouped", groups=bundle.model_data["garage_id"]
)
thresholds = threshold_report(oof)
thresholds


,threshold,precision,recall,flag_rate,pr_auc
0,0.1,0.033616,0.952041,0.862599,0.045808
1,0.2,0.033650,0.774490,0.701019,0.045808
2,0.3,0.033138,0.594898,0.546774,0.045808
3,0.4,0.034392,0.455102,0.403033,0.045808
4,0.5,0.037029,0.340816,0.280333,0.045808


In [10]:
imbalance_comparison.to_csv(OUT / "imbalance_comparison.csv", index=False)
thresholds.to_csv(OUT / "threshold_report.csv", index=False)


## 4. Imbalance comparison

The project requires explicit comparison of at least two imbalance approaches and their effects on precision and recall. The reusable evaluation function reports precision/recall at the default 0.5 threshold. Threshold adjustment should be assessed separately at the selected operating threshold because the project ultimately requires a cost-based operating point.

## 5. Production-data limitation

The brief describes a July–December 2026 final holdout, but the supplied claims file currently extends only to August 2026. Therefore the available July-onward records can be treated as the available final-holdout slice, but the team must not describe it as a complete July–December holdout.